# Oil Well Profitability Analysis

## Introduction

OilyGiant Mining Company is evaluating three candidate regions for developing 200 new oil wells. The goal of this project is to identify which region will yield the highest profit while keeping the risk of loss below 2.5%.

**Approach:**
1. Train a Linear Regression model on geological survey data for each region to predict oil reserve volume per well.
2. Use predictions to select the top 200 wells from a random sample of 500, and estimate profit.
3. Apply bootstrapping (1,000 simulations) to build a profit distribution and assess financial risk for each region.
4. Recommend the region that maximizes expected profit within an acceptable risk threshold.

## Dataset

Three datasets are provided — one per region (`geo_data_0.csv`, `geo_data_1.csv`, `geo_data_2.csv`). Each contains 100,000 oil well records with the following features:

| Column | Description |
|---|---|
| `id` | Unique well identifier |
| `f0`, `f1`, `f2` | Geological survey features |
| `product` | Target: oil reserve volume (thousand barrels) |

**Business constraints:**
- Total development budget: **\$100M**
- Wells to develop per region: **200**
- Revenue per thousand barrels: **\$4,500**
- Break-even volume per well: **111.1 thousand barrels**

## 1. Data Loading & Exploration

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

# Load the datasets
data_0 = pd.read_csv('/datasets/geo_data_0.csv')
data_1 = pd.read_csv('/datasets/geo_data_1.csv')
data_2 = pd.read_csv('/datasets/geo_data_2.csv')

# Preview the first few rows of each dataset
print("Region 0:")
print(data_0.head())
print("\nRegion 1:")
print(data_1.head())
print("\nRegion 2:")
print(data_2.head())

Region 0:
      id        f0        f1        f2     product
0  txEyH  0.705745 -0.497823  1.221170  105.280062
1  2acmU  1.334711 -0.340164  4.365080   73.037750
2  409Wp  1.022732  0.151990  1.419926   85.265647
3  iJLyR -0.032172  0.139033  2.978566  168.620776
4  Xdl7t  1.988431  0.155413  4.751769  154.036647

Region 1:
      id         f0         f1        f2     product
0  kBEdx -15.001348  -8.276000 -0.005876    3.179103
1  62mP7  14.272088  -3.475083  0.999183   26.953261
2  vyE1P   6.263187  -5.948386  5.001160  134.766305
3  KcrkZ -13.081196 -11.506057  4.999415  137.945408
4  AHL4O  12.702195  -8.147433  5.004363  134.766305

Region 2:
      id        f0        f1        f2     product
0  fwXo0 -1.146987  0.963328 -0.828965   27.758673
1  WJtFt  0.262778  0.269839 -2.530187   56.069697
2  ovLUW  0.194587  0.289035 -5.586433   62.871910
3  q6cA6  2.236060 -0.553760  0.930038  114.572842
4  WPMUX -0.515993  1.716266  5.899011  149.600746


## 2. Feature Engineering

The `id` column is a non-predictive identifier and is dropped. The `product` column is isolated as the target variable for each region.

In [2]:
# Separate features and target for each region
features_0 = data_0.drop(['id', 'product'], axis=1)
target_0 = data_0['product']

features_1 = data_1.drop(['id', 'product'], axis=1)
target_1 = data_1['product']

features_2 = data_2.drop(['id', 'product'], axis=1)
target_2 = data_2['product']

## 3. Model Training & Evaluation

A Linear Regression model is trained for each region using a 75/25 train/validation split. Model quality is measured by **RMSE** (Root Mean Squared Error) — lower values indicate more accurate volume predictions.

### 3a. Region 0

In [3]:
# Split data into training and validation sets
features_train_0, features_valid_0, target_train_0, target_valid_0 = train_test_split(
    features_0, target_0, test_size=0.25, random_state=12345
)

# Train the model
model_0 = LinearRegression()
model_0.fit(features_train_0, target_train_0)

# Make predictions
predictions_0 = model_0.predict(features_valid_0)

# Evaluate the model
rmse_0 = mean_squared_error(target_valid_0, predictions_0, squared=False)
print("Region 0 - RMSE:", rmse_0)
print("Region 0 - Mean predicted volume:", predictions_0.mean())

Region 0 - RMSE: 37.5794217150813
Region 0 - Mean predicted volume: 92.59256778438035


Region 0 has a relatively high RMSE of ~37.6, meaning predictions vary considerably from actual values. The mean predicted volume of ~92.6 thousand barrels falls below the break-even threshold of 111.1, so careful well selection will be essential.

### 3b. Region 1

In [4]:
# Split data into training and validation sets
features_train_1, features_valid_1, target_train_1, target_valid_1 = train_test_split(
    features_1, target_1, test_size=0.25, random_state=12345
)

# Train the model
model_1 = LinearRegression()
model_1.fit(features_train_1, target_train_1)

# Make predictions
predictions_1 = model_1.predict(features_valid_1)

# Evaluate the model
rmse_1 = mean_squared_error(target_valid_1, predictions_1, squared=False)
print("Region 1 - RMSE:", rmse_1)
print("Region 1 - Mean predicted volume:", predictions_1.mean())

Region 1 - RMSE: 0.893099286775617
Region 1 - Mean predicted volume: 68.728546895446


Region 1 stands out with an exceptionally low RMSE of ~0.89 — the model predicts reserve volumes with very high accuracy. Although the mean predicted volume (~68.7) appears low, the model's precision means the top wells identified will be highly reliable picks.

### 3c. Region 2

In [5]:
# Split data into training and validation sets
features_train_2, features_valid_2, target_train_2, target_valid_2 = train_test_split(
    features_2, target_2, test_size=0.25, random_state=12345
)

# Train the model
model_2 = LinearRegression()
model_2.fit(features_train_2, target_train_2)

# Make predictions
predictions_2 = model_2.predict(features_valid_2)

# Evaluate the model
rmse_2 = mean_squared_error(target_valid_2, predictions_2, squared=False)
print("Region 2 - RMSE:", rmse_2)
print("Region 2 - Mean predicted volume:", predictions_2.mean())

Region 2 - RMSE: 40.02970873393434
Region 2 - Mean predicted volume: 94.96504596800489


Region 2 has the highest RMSE (~40.0) of the three regions, indicating the most uncertain predictions. Its mean predicted volume (~95.0) is also below the break-even threshold, making risk assessment especially important here.

## 4. Break-Even Analysis

Before estimating profits, we calculate the minimum oil volume per well needed for the development to break even given the fixed budget.

In [6]:
# Business constants
BUDGET = 100_000_000          # USD
WELL_COUNT = 200
REVENUE_PER_BARREL = 4.5 * 1000  # USD per thousand barrels

# Cost per well
WELL_COST = BUDGET / WELL_COUNT

# Minimum volume needed to break even
MIN_VOLUME = WELL_COST / REVENUE_PER_BARREL

print("Minimum volume needed to break even (thousand barrels):", MIN_VOLUME)

Minimum volume needed to break even (thousand barrels): 111.11111111111111


Each well must produce at least **111.1 thousand barrels** to recover its development cost. Since all three regions have mean predicted volumes below this threshold, profitability depends entirely on selecting the highest-yielding wells from each region — which is the purpose of the selection strategy below.

## 5. Profit Calculation

The profit function simulates a realistic selection process: from a random sample of 500 candidate wells, the top 200 by predicted volume are chosen for development. Actual (not predicted) volumes are used to compute realized profit.

In [7]:
def calculate_profit(predictions, actual, sample_size=500):
    # Take a random sample of 500 wells
    sample_indices = np.random.choice(predictions.index, size=sample_size, replace=False)
    sample_predictions = predictions.loc[sample_indices]
    sample_actual = actual.loc[sample_indices]
    
    # Select top 200 from that sample by predicted volume
    top_200_indices = sample_predictions.sort_values(ascending=False).index[:200]
    total_product = sample_actual.loc[top_200_indices].sum()
    
    # Calculate profit
    profit = total_product * REVENUE_PER_BARREL - BUDGET
    return profit

In [8]:
# Convert predictions to Series for indexing
predictions_0_series = pd.Series(predictions_0, index=target_valid_0.index)
predictions_1_series = pd.Series(predictions_1, index=target_valid_1.index)
predictions_2_series = pd.Series(predictions_2, index=target_valid_2.index)

# Single profit estimate per region
profit_0 = calculate_profit(predictions_0_series, target_valid_0)
profit_1 = calculate_profit(predictions_1_series, target_valid_1)
profit_2 = calculate_profit(predictions_2_series, target_valid_2)

print("Region 0 - Estimated profit (USD):", profit_0)
print("Region 1 - Estimated profit (USD):", profit_1)
print("Region 2 - Estimated profit (USD):", profit_2)

Region 0 - Estimated profit (USD): 3528929.4132888913
Region 1 - Estimated profit (USD): 5540676.888422787
Region 2 - Estimated profit (USD): 730390.5535787791


## 6. Bootstrap Risk Analysis

A single profit estimate is insufficient for a decision of this magnitude. Bootstrap resampling (1,000 simulations) is used to build a full distribution of possible profits for each region, enabling a statistical assessment of expected return and downside risk.

In [9]:
def bootstrap_profit(predictions, actual, n_simulations=1000, sample_size=500):
    state = np.random.RandomState(12345)
    profits = []

    for _ in range(n_simulations):
        # Random sample of 500 wells with replacement
        sample_indices = state.choice(predictions.index, size=sample_size, replace=True)
        sample_predictions = predictions.loc[sample_indices]
        sample_actual = actual.loc[sample_indices]

        # Select top 200 predicted wells
        top_200_indices = sample_predictions.sort_values(ascending=False).index[:200]
        total_product = sample_actual.loc[top_200_indices].sum()

        # Calculate profit
        profit = total_product * REVENUE_PER_BARREL - BUDGET
        profits.append(profit)

    return pd.Series(profits)

In [10]:
# Bootstrap analysis — Region 0
profits_0 = bootstrap_profit(predictions_0_series, target_valid_0)
mean_0 = profits_0.mean()
conf_int_0 = profits_0.quantile([0.025, 0.975])
risk_0 = (profits_0 < 0).mean() * 100

print("Region 0 - Mean profit:", mean_0)
print("Region 0 - 95% confidence interval:", conf_int_0)
print("Region 0 - Risk of loss (%):", risk_0)

# Bootstrap analysis — Region 1
profits_1 = bootstrap_profit(predictions_1_series, target_valid_1)
mean_1 = profits_1.mean()
conf_int_1 = profits_1.quantile([0.025, 0.975])
risk_1 = (profits_1 < 0).mean() * 100

print("\nRegion 1 - Mean profit:", mean_1)
print("Region 1 - 95% confidence interval:", conf_int_1)
print("Region 1 - Risk of loss (%):", risk_1)

# Bootstrap analysis — Region 2
profits_2 = bootstrap_profit(predictions_2_series, target_valid_2)
mean_2 = profits_2.mean()
conf_int_2 = profits_2.quantile([0.025, 0.975])
risk_2 = (profits_2 < 0).mean() * 100

print("\nRegion 2 - Mean profit:", mean_2)
print("Region 2 - 95% confidence interval:", conf_int_2)
print("Region 2 - Risk of loss (%):", risk_2)

Region 0 - Mean profit: 6007352.442611653
Region 0 - 95% confidence interval: 0.025    1.294833e+05
0.975    1.231164e+07
dtype: float64
Region 0 - Risk of loss (%): 2.0

Region 1 - Mean profit: 6652410.582210722
Region 1 - 95% confidence interval: 0.025    1.579885e+06
0.975    1.197642e+07
dtype: float64
Region 1 - Risk of loss (%): 0.3

Region 2 - Mean profit: 6155597.228409678
Region 2 - 95% confidence interval: 0.025   -1.221850e+05
0.975    1.230644e+07
dtype: float64
Region 2 - Risk of loss (%): 3.0


## 7. Profit Distribution Visualization

Histograms of the 1,000 simulated profit outcomes illustrate the spread and shape of the profit distribution for each region, making it easier to visualize tail risk and central tendency side by side.

In [11]:
def plot_profit_distribution(profits, region_label):
    plt.figure(figsize=(10, 6))
    plt.hist(profits, bins=50, color='skyblue', edgecolor='black')
    plt.title(f'Profit Distribution - {region_label}')
    plt.xlabel('Profit (USD)')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.show()

# Plot profit distribution for each region
plot_profit_distribution(profits_0, "Region 0")
plot_profit_distribution(profits_1, "Region 1")
plot_profit_distribution(profits_2, "Region 2")

## 8. Conclusion & Recommendation

### Summary of Results

| Region | Model RMSE | Mean Profit | 95% CI Lower | 95% CI Upper | Risk of Loss |
|---|---|---|---|---|---|
| Region 0 | 37.58 | \$6.01M | \$129K | \$12.31M | 2.0% |
| **Region 1** | **0.89** | **\$6.65M** | **\$1.58M** | **\$11.98M** | **0.3%** |
| Region 2 | 40.03 | \$6.16M | -\$122K | \$12.31M | 3.0% |

### Recommendation: Region 1

**Region 1 is the clear recommendation** based on both profitability and risk:

- **Highest expected profit** — Mean of \$6.65M across 1,000 simulations, outperforming both other regions.
- **Lowest risk of loss** — Only 0.3% of simulations resulted in a loss, well below the 2.5% threshold. Regions 0 (2.0%) and 2 (3.0%) are both closer to or exceeding the acceptable limit.
- **Most reliable model** — An RMSE of just 0.89 means the Linear Regression model predicts oil volumes with exceptional accuracy for Region 1. This translates directly into more reliable well selection and less profit variability.
- **Strongest downside protection** — Region 1's 95% confidence interval lower bound is \$1.58M, meaning even in adverse scenarios the region remains profitable. Region 2's lower bound dips negative (-\$122K), confirming meaningful loss exposure.

Regions 0 and 2 both show higher prediction uncertainty and elevated risk profiles, making them less suitable for a capital-intensive development commitment of this scale.